# Phi-4-mini-instruct — descriptions (retrieval-aligned re-run)

Corrected re-run of `source code/01_251029_generating_description_Phi-4-mini-Instructt.ipynb`.
The retrieval stack (embedding model + PDF reader) and `max_new_tokens` are aligned to the
Apertus-8B and GPT-5-nano notebooks so the three runs compare **models**, not pipelines.

Changed vs original: `bge-small-en-v1.5` -> `BAAI/bge-m3`; `pypdf` -> `PyMuPDFReader`;
`max_new_tokens` 500 -> 256; added `top_p=0.9`; isolated persist/output dirs.
Kept (Phi-4 format requirements): 4-bit quantization, Phi chat template, `stopping_ids`, `context_window=4096`.

Run on the cluster, e.g.:

```
sbatch --gpus=2 --gres=gpumem:40g --time=05:00:00 --mem-per-cpu=32g \
  --wrap="jupyter nbconvert --to notebook --execute 01_260603_phi4-mini_aligned_generating_description.ipynb --inplace"
```

In [1]:
import os

# --- CONFIG (paths routed through env vars; defaults assume this folder is a
#     sibling of the cluster `data/`, `models/`, `storage/` folders) ---
INPUT_XLSX     = os.environ.get("INPUT_XLSX", "./data/251027 input data points and groups.xlsx")
LITERATURE_DIR = os.environ.get("LITERATURE_DIR", "../../data/literature")
PERSIST_DIR    = os.environ.get("PERSIST_DIR", "../storage/phi4_aligned")
RESPONSE_DIR   = os.environ.get("RESPONSE_DIR", "./260603_generated_descriptions_Phi-4-mini-aligned")

# --- Models (retrieval aligned to Apertus / GPT-5-nano) ---
EMBED_MODEL = os.environ.get("EMBED_MODEL", "models/BAAI-bge-m3")          # was ../models/BAAI-bge-small-en-v1.5
LLM_MODEL   = "models/Phi-4-mini-instruct"

# Generation-side knobs
USE_QUANTIZATION = True   # set False for full-precision parity with Apertus (needs more GPU memory)
MAX_NEW_TOKENS   = 256    # was 500; aligned to the other two notebooks

os.makedirs(RESPONSE_DIR, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)

In [2]:
import pandas as pd

initial_groups_df = pd.read_excel(INPUT_XLSX)
initial_groups_df['Data_groups'] = initial_groups_df['Data_groups']\
    .str.strip().str.lower().str.replace('&', 'and')
initial_groups_df.head()

,Data_groups,Data_points,Source,File,Passport_type
0,general information,"Product commercial name, Manufacturer's name, ...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,material health,"Security information, warnings, recommendation...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,sustainability,"Environmental declaration, Life cycle assessme...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,design and production,"Manufacturing process, Manufacturing technique...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,use and operate phase,"Positioning in the building, location in the b...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [3]:
initial_points_df = initial_groups_df.assign(Data_points=initial_groups_df['Data_points']\
    .str.replace('&', 'and').str.split(',')).explode('Data_points').reset_index()
initial_points_df['Data_points'] = initial_points_df['Data_points']\
    .str.strip().str.lower()

initial_points_df = initial_points_df[initial_points_df["Data_points"] != ""].reset_index(drop=True)

initial_points_df.head()

,index,Data_groups,Data_points,Source,File,Passport_type
0,0,general information,product commercial name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,0,general information,manufacturer's name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,0,general information,manufacturer's details,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,0,general information,materials composition,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,0,general information,product properties,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [4]:
initial_points_df['description'] = ""
initial_points_df.shape

(2010, 7)

In [5]:
import os

count = 0
files = [f for f in os.listdir(LITERATURE_DIR) if os.path.isfile(os.path.join(LITERATURE_DIR, f))]

for source in initial_points_df.File.unique():
    if os.path.isfile(os.path.join(LITERATURE_DIR, source)):
        count += 1
    else:
        print("Missing source file!", source)
count, initial_points_df.File.unique().shape, len(files)

(30, (30,), 30)

In [6]:
import logging
import sys
import torch

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [7]:
from llama_index.core import (
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader,
    PromptTemplate,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import BitsAndBytesConfig

/cluster/home/svangelova/Pre-demolition audits/env/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [8]:
# --- Embeddings (ALIGNED: bge-m3, same as Apertus / GPT-5-nano) ---
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: models/BAAI-bge-m3


Load pretrained SentenceTransformer: models/BAAI-bge-m3


In [9]:
# --- LLM: Phi-4-mini-instruct (generation kept Phi-4-specific; sampling aligned to Apertus) ---
system_prompt = """<|system|>
You are a helpful assistant with expertise in sustainable construction.

Rules:
1. Use exactly three sentences.
2. Base your answer on the provided text.
3. Do not add extra explanations, commentary, or unrelated information.<|end|>
"""

# Phi-4 chat template wrapper (kept as-is)
query_wrapper_prompt = PromptTemplate("<|USER|>\n{query_str}\n<|ASSISTANT|>")

model_kwargs = {}
if USE_QUANTIZATION:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=MAX_NEW_TOKENS,                       # ALIGNED: 256
    generate_kwargs={"temperature": 0.1,
                     "do_sample": True,                  # sampling required for temperature
                     "top_p": 0.9},                      # ALIGNED: added to match Apertus
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name=LLM_MODEL,
    model_name=LLM_MODEL,
    device_map="auto",
    stopping_ids=[32000, 32001, 32007],                 # Phi-4 stop tokens (kept)
    tokenizer_kwargs={"max_length": 4096},
    model_kwargs=model_kwargs,
)

Settings.llm = llm

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
# --- Load PDFs (ALIGNED: PyMuPDFReader, same as Apertus / GPT-5-nano -> page-level docs) ---
from pathlib import Path
from llama_index.readers.file import PyMuPDFReader

pdf_reader = {".pdf": PyMuPDFReader()}
reader = SimpleDirectoryReader(
    input_dir=Path(LITERATURE_DIR),
    file_extractor=pdf_reader,
)
documents = reader.load_data()

print(f"Loaded {len(documents)} documents.")
print(documents[0].text[:500])

Loaded 1012 documents.
Journal of Building Engineering 43 (2021) 103233
Available online 3 September 2021
2352-7102/© 2021 Elsevier Ltd. All rights reserved.
Digitizing material passport for sustainable construction projects using BIM 
Islam Atta a, Emad S. Bakhoum b,c,*, Mohamed M. Marzouk d 
a Teaching Assistant, Civil Engineering Department, Al-Madina Higher Institute for Engineering and Technology, Giza, Egypt 
b Civil Infrastructure Engineering and Management Department, Nile University, Giza, Egypt 
c Civil Engi


In [11]:
# --- Build index (with bge-m3) + retrieval engine ---
index = VectorStoreIndex.from_documents(documents)
index.storage_context.persist(persist_dir=PERSIST_DIR)

query_engine = index.as_query_engine(response_mode="compact")  # default similarity_top_k=2

In [12]:
import json

descriptions = []
descriptions_per_point = {}
responses = {}
all_references = []

for i in range(initial_points_df.shape[0]):
    if os.path.exists(f"{RESPONSE_DIR}/{i}_response.json"):
        continue
    file = initial_points_df.iloc[i].File
    data_group = initial_points_df.iloc[i].Data_groups
    data_point = initial_points_df.iloc[i].Data_points

    prompt = f"What is the meaning of {data_point} in relation to {data_group}?"
    response = query_engine.query(prompt)

    print(f'index: {i}')

    point_i_response = {}
    point_i_response["question"] = prompt
    point_i_response["original_source"] = file
    point_i_response["data_group"] = data_group
    point_i_response["data_point"] = data_point

    references = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node
        reference = {}
        reference["text"] = node.get_text()
        reference["metadata"] = node.metadata
        reference["score"] = node_with_score.score
        references.append(reference)

    point_i_response["reference_1"] = references[0]["metadata"]["file_name"]
    point_i_response["reference_2"] = references[1]["metadata"]["file_name"]
    point_i_response["description"] = response.response
    point_i_response["references"] = references

    with open(f"{RESPONSE_DIR}/{i}_response.json", "w") as f:
        json.dump(point_i_response, f)

    descriptions.append(response.response)
    descriptions_per_point[data_point] = response.response
    responses[i] = response
    initial_points_df.loc[i, 'description'] = response.response

    all_references.append(point_i_response["reference_1"])
    all_references.append(point_i_response["reference_2"])

index: 0


index: 1


index: 2


index: 3


index: 4


...

index: 2005


index: 2006


index: 2007


index: 2008


index: 2009


In [13]:
from collections import Counter
Counter(all_references)

Counter({'CPR 2024.pdf': 714,
         'BAMB 2019.pdf': 486,
         'ESPR 2024.pdf': 453,
         'ISO 59040 2025.pdf': 376,
         'Seddiqui 2024.pdf': 251,
         'Bosma 2024.pdf': 240,
         'Kebede 2024.pdf': 223,
         'Van Capelleveen 2023.pdf': 172,
         'Platform CB 2023.pdf': 144,
         'Jensen 2023.pdf': 130,
         'Circularise 2025.pdf': 105,
         'Christensen 2025.pdf': 84,
         'Heisel and Rau-Oberhuber 2020.pdf': 81,
         'Mao and Cao 2025.pdf': 81,
         'Atta 2021.pdf': 72,
         'Giovanardi 2023.pdf': 64,
         'Stratmann 2023.pdf': 57,
         'Wan and Jiang 2025.pdf': 42,
         'Bauen digital Schweiz 2024.pdf': 41,
         'Honic 2019.pdf': 40,
         'Honic 2021.pdf': 33,
         'Markou 2025.pdf': 28,
         'Mulhall 2022.pdf': 26,
         'Çetin 2023.pdf': 17,
         'Ruismäki 2025.pdf': 15,
         'Göswein 2022.pdf': 15,
         'Munaro and Tavares 2021.pdf': 11,
         'Byers 2025.pdf': 9,
         'K

In [14]:
for file in files:
    if file not in Counter(all_references).keys():
        print(file)